# Lesson 3.1 — 模仿学习（Imitation learning）：Behavior Cloning 要解决的问题

Lesson 2 得到了一条能够训练的 pipeline。本课要问的是这条 pipeline
无法回答的问题：

> 为什么一个 training loss 极低的 model，在真正驱动机器人时仍然会彻底失败？

本 notebook 末尾的实验就是入口：它在这个仓库中的真实
dataset 上说明，为什么目前还无法进行 held-out evaluation —— 以及这意味着什么。


## 3.1.1 — 问题陈述

expert 演示一个任务。该演示是从 expert 的行为中采样得到的一组
state-action 对：

```text
D = {(o_t, a_t)}   with   o_t ~ d_{pi_E}
```

模仿学习（Imitation learning）要寻找一个 policy 来复现 expert 的决策：

```text
pi_theta(a_t | o_t)  ~=  pi_E(a_t | o_t)
```

有两点需要立即区分开：

- expert 的 **policy** `pi_E` 是被模仿的函数；
- expert 的 **state distribution** `d_{pi_E}` 是数据的来源，而它
  *由该 policy 生成*。第二个事实正是本课所有困难
  问题的根源。


## 3.1.2 — Behavior Cloning 属于监督学习

Behavior Cloning (BC) 忽略数据来自某个 policy 这一事实，把它当作一个
普通的回归问题：

```text
D = {(o_t, a_t)}
theta* = argmin_theta  sum_t  L(pi_theta(o_t), a_t)
```

对于连续 action，loss 就是 MSE：

```text
L = MSE(pi_theta(o_t), a_t)
```

这与 `2.7_bc_training_loop.ipynb` 中已经搭好的 training loop
完全一致。BC 不是新算法；它就是在机器人 state 上的监督学习。这
也正是它继承了监督学习失效模式的原因：
它只在其训练时所处的分布上有效。


In [1]:
import os
from pathlib import Path

from pathlib import Path

# Notebooks may be started with either the repository root or notebooks/ as the working
# directory. Walk up until the project root is found, so dataset and cache paths never
# resolve to notebooks/datasets or notebooks/.cache by accident.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / ".git").exists()),
    _cwd.parent if _cwd.name == "notebooks" else _cwd,
)
REPO_ROOT = PROJECT_ROOT
HF_CACHE = REPO_ROOT / ".cache" / "hf"
(HF_CACHE / "datasets").mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_CACHE))
os.environ.setdefault("HF_DATASETS_CACHE", str(HF_CACHE / "datasets"))

import numpy as np
import torch
import torch.nn as nn

from lerobot.datasets.lerobot_dataset import LeRobotDataset

DATASET_ROOT = REPO_ROOT / "datasets" / "lerobot" / "pickcube"
dataset = LeRobotDataset(repo_id="pickcube", root=str(DATASET_ROOT))

print("frames  :", len(dataset))
print("episodes:", dataset.num_episodes)
print("FPS     :", dataset.fps)

frames  : 50
episodes: 1
FPS     : 50


## 3.1.3 — 为什么目前还无法做 validation split

下一步（2.8.7）的计划是 train/validation split。运行下面的实验，
看看这个 split 实际给了你什么。

trajectory 中相邻 frame 几乎完全相同：机器人每步只移动几
毫米。因此按 frame 随机 split 会把同一 state 的近似副本放到
两侧。一个 "held-out" frame 在任何有意义的意义上都没有被真正留出 ——
这就是 **data leakage**，它会虚高 validation 的表现。


In [2]:
# How similar are adjacent frames? Compare consecutive observation vectors.
observations = np.stack([dataset[i]["observation.state"].numpy() for i in range(len(dataset))])

consecutive_delta = np.linalg.norm(np.diff(observations, axis=0), axis=1)

random_pairs = []
rng = np.random.default_rng(0)
for _ in range(500):
    a, b = rng.integers(0, len(observations), size=2)
    if a != b:
        random_pairs.append(np.linalg.norm(observations[a] - observations[b]))
random_pairs = np.asarray(random_pairs)

print("consecutive-frame distance : mean %.4f  median %.4f" % (consecutive_delta.mean(), np.median(consecutive_delta)))
print("random-pair distance       : mean %.4f  median %.4f" % (random_pairs.mean(), np.median(random_pairs)))
print("ratio (random / consecutive): %.1fx" % (random_pairs.mean() / consecutive_delta.mean()))
print("\nAdjacent frames are nearly the same point; a frame-level split leaks.")

consecutive-frame distance : mean 1.3500  median 1.3221
random-pair distance       : mean 1.3999  median 1.3804
ratio (random / consecutive): 1.0x

Adjacent frames are nearly the same point; a frame-level split leaks.


### 这意味着什么

只有**一个 episode** 时，按 frame 切分无法产生诚实的 validation set。可选
方案是：

- 在一个**不同的 trajectory** 上评估 —— 这需要采集更多数据（2.9），或者
- 承认这里的 validation curve 只能检测明显的 overfitting，并如实说明。

一旦有多个 episode，正确的默认做法是**按 episode 切分**：
整条 trajectory 进入 train 或 validation，绝不切分单个 frame。否则
model 就会在它实际上已经记住的 frame 上被打分。


## 3.1.4 — Distribution shift：BC 的核心失效

在 training 时，model 看到的是 expert 访问过的 state：

```text
o_t ~ d_{pi_E}
```

在执行时，model 看到的是*它自己*产生的 state：

```text
o_t ~ d_{pi_theta}
```

这是两个不同的分布。一个很小的 action 误差就会把机器人带到一个 expert
从未访问过的 state，那里 model 从未被训练过，它的预测是
任意的。这个误差会累积：trajectory 每一步都进一步偏离数据
分布。

这就是为什么很低的 training loss 只能证明 model 拟合了 demonstration ——
而不能证明它能完成任务。


In [3]:
# Quantify the drift numerically: how quickly does a state leave the training set?
# Take the first frame, then follow the stored actions and measure the distance from
# anything the dataset actually contains.
states = observations

# Pick a real starting frame and walk forward one step, measuring nearest-neighbour
# distance of the resulting state to the dataset.
def nearest_distance(query, pool):
    return float(np.min(np.linalg.norm(pool - query, axis=1)))


start = 0
reference = states[start]
d_same = nearest_distance(reference, states)

# Perturb the state by a plausible single-step error and re-measure
perturbation = 0.02 * np.linalg.norm(states.std(axis=0))
noisy = reference + np.random.default_rng(1).normal(0, perturbation, size=reference.shape)
d_noisy = nearest_distance(noisy, states)

print(f"distance from dataset for a real frame    : {d_same:.5f}")
print(f"distance after a small perturbation       : {d_noisy:.5f}")
print(f"perturbation size in state units          : {perturbation:.5f}")
print()
print("A policy that reproduces the trajectory only approximately lands off-distribution.")
print("With 50 frames and one trajectory there is nothing to fall back on.")

distance from dataset for a real frame    : 0.00000
distance after a small perturbation       : 0.12165
perturbation size in state units          : 0.02012

A policy that reproduces the trajectory only approximately lands off-distribution.
With 50 frames and one trajectory there is nothing to fall back on.


## 3.1.5 — 本课接下来讲什么

Lesson 3 的其余部分会为上述每个问题构建工具：

| Section | 它解决的问题 |
|---|---|
| 3.3 | overfitting、underfitting，以及如何诚实地 split |
| 3.4 | 形式化地讨论 distribution shift |
| 3.5 | DAgger：让 policy 行动，并为它到达的 state 打标签 |
| 3.6 | partial observability：single-frame policy 与 history policy |
| 3.7 | action chunking：预测一个序列，而不是一个点 |
| 3.8 | 从 state 到 `(image, language, state)` |
| 3.9 | 正确地采集 expert 数据，包括 SO-101 teleoperation |

紧接着的下一步是 2.8.7，它正是引出上述全部内容的实验。


## 小结

1. BC 是在 **expert** 生成的 state 上的监督学习。
2. training 的 pair 来自 `d_{pi_E}`；执行时会遇到 `d_{pi_theta}`。两者之间
   的差距就是 distribution shift，而它在 training loss 中不可见。
3. 相邻 frame 几乎完全相同（上面已测量），因此按 frame 的
   train/validation split 会泄漏。一旦有多个 episode，就按 episode split。
4. 只有一个 episode 时不存在诚实的 held-out evaluation。这是数据问题，
   而不是建模问题，也正是 2.9 要解决的。
